In [1]:
pip install transformers datasets torch pandas scikit-learn accelerate

Note: you may need to restart the kernel to use updated packages.


In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "GroNLP/hateBERT"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4
)

C:\Users\dj140\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `2`.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14215.07it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: GroNLP/hateBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECT

In [2]:
from pathlib import Path

main_dir = Path.cwd()
print(main_dir)

c:\Users\dj140\OneDrive\Desktop\project\MSc-DSA-2026-Thesis-Project\hate_bert_experiment


# Using CONDA

In [3]:
import pandas as pd

df = pd.read_csv(main_dir / "CONDA" / "CONDA_train.csv" )

In [4]:
print(df)

          Id  matchId  conversationId               utterance  chatTime  \
0      11263      697            3193                    wow!        76   
1      13741      843            3809                     WTF      1563   
2      22125     1412            6199                 wpe wpe      2853   
3       6453      439            1875                  hahaha      1038   
4       9644      601            2713                     wtf      1661   
...      ...      ...             ...                     ...       ...   
26916  11284      699            3201         cant believe it      3029   
26917  44732     3028           12854                      AH      1836   
26918  38158     2496           10711                       !      2171   
26919    860       72             289  dayuuuuuum [SEPA] lool      1721   
26920  15795      977            4349                      Gg      2285   

       playerSlot                     playerId intentClass slotClasses  \
0               0      AN

In [5]:
label_map = {
    "E": 0,
    "I": 1,
    "A": 2,
    "O": 3
}

df["label"] = df["intentClass"].map(label_map)

df = df[[ "utterance","label"]]

df.tail()

,utterance,label
26916,cant believe it,3
26917,AH,3
26918,!,3
26919,dayuuuuuum [SEPA] lool,3
26920,Gg,3


In [7]:
df[df["utterance"].apply(lambda x: not isinstance(x, str))].head()

,utterance,label
701,NaN,3
7720,NaN,3
8623,NaN,3
12751,NaN,3
15936,NaN,3


In [8]:
df["utterance"] = df["utterance"].astype(str)
df = df[df["utterance"].str.strip() != ""]

In [9]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)

dataset = dataset.train_test_split(
    test_size=0.2,
    seed=42
)

dataset

DatasetDict({
    train: Dataset({
        features: ['utterance', 'label'],
        num_rows: 21536
    })
    test: Dataset({
        features: ['utterance', 'label'],
        num_rows: 5385
    })
})

In [10]:
def tokenize(batch):

    return tokenizer(
        batch["utterance"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

In [11]:
tokenized_dataset = dataset.map(
    tokenize,
    batched=True
)

Map: 100%|██████████| 5385/5385 [00:00<00:00, 13959.67 examples/s]


In [12]:
from transformers import TrainingArguments


training_args = TrainingArguments(
    output_dir="./hateBERT-CONDA",

    num_train_epochs=3,

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True
)

In [13]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np


def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=1
    )

    return {
        "accuracy":
            accuracy_score(labels, predictions),

        "f1":
            f1_score(
                labels,
                predictions,
                average="macro"
            )
    }

In [ ]:
from transformers import Trainer


trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=
        tokenized_dataset["train"],

    eval_dataset=
        tokenized_dataset["test"],

    compute_metrics=
        compute_metrics
)


trainer.train()

In [28]:
import torch

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)
model.eval()

text = "he reported me"

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}


with torch.no_grad():
    output = model(**inputs)


probabilities = torch.softmax(
    output.logits,
    dim=1
)

pred = torch.argmax(
    probabilities,
    dim=1
).item()

print(probabilities)
print([key for key, val in label_map.items() if val == pred])

tensor([[0.0125, 0.0071, 0.9701, 0.0103]], device='cuda:0')
['A']


# Using GameTox

In [15]:
tokenizer_two = AutoTokenizer.from_pretrained(MODEL_NAME)
model_two = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=6
)

[transformers] You passed `num_labels=6` which is incompatible to the `id2label` map of length `2`.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3450.25it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: GroNLP/hateBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider trainin

In [16]:
df = pd.read_csv(main_dir / "GameTox" / "train.csv" )
print(df)

       index                                            message  label
0      30702                                            no rush    0.0
1      18607                      whatever ... watch the replay    0.0
2      32901                                            useless    1.0
3      25964                                          3 gunmark    0.0
4      28643                                                lol    0.0
...      ...                                                ...    ...
42954  14410                                              давай    0.0
42955  50058                             i had over 7k combined    0.0
42956  10745  like move your tank into the fight instead of ...    0.0
42957  37073                           ihr scheiss juden kinder    0.0
42958  22336                                               omfg    2.0

[42959 rows x 3 columns]


In [17]:
df["utterance"] = df["message"]
df["label"] = df["label"].astype(int)
df = df[["utterance", "label"]]

In [18]:
df.tail()

,utterance,label
42954,давай,0
42955,i had over 7k combined,0
42956,like move your tank into the fight instead of ...,0
42957,ihr scheiss juden kinder,0
42958,omfg,2


In [19]:
df[df["utterance"].apply(lambda x: not isinstance(x, str))].head()

,utterance,label


In [20]:
dataset = Dataset.from_pandas(df)

dataset = dataset.train_test_split(
    test_size=0.2,
    seed=42
)

dataset

DatasetDict({
    train: Dataset({
        features: ['utterance', 'label'],
        num_rows: 34367
    })
    test: Dataset({
        features: ['utterance', 'label'],
        num_rows: 8592
    })
})

In [21]:
def tokenize(batch):

    return tokenizer_two(
        batch["utterance"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

In [22]:
tokenized_dataset = dataset.map(
    tokenize,
    batched=True
)

Map: 100%|██████████| 8592/8592 [00:00<00:00, 17991.97 examples/s]


In [24]:
training_args = TrainingArguments(
    output_dir="./hateBERT-GameTox",

    num_train_epochs=3,

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True
)

In [25]:
trainer = Trainer(
    model=model_two,

    args=training_args,

    train_dataset=
        tokenized_dataset["train"],

    eval_dataset=
        tokenized_dataset["test"],

    compute_metrics=
        compute_metrics
)


trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.327813,0.290823,0.908520,0.375722
2,0.243628,0.319422,0.908403,0.477408
3,0.216963,0.348610,0.905843,0.495756


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.47it/s]


TrainOutput(global_step=6444, training_loss=0.2757348144076612, metrics={'train_runtime': 2199.8809, 'train_samples_per_second': 46.867, 'train_steps_per_second': 2.929, 'total_flos': 6781996792171008.0, 'train_loss': 0.2757348144076612, 'epoch': 3.0})

In [47]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_two.to(device)
model_two.eval()

text = "you are terrible lol"

inputs = tokenizer_two(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}


with torch.no_grad():
    output = model_two(**inputs)


probabilities = torch.softmax(
    output.logits,
    dim=1
)

pred = torch.argmax(
    probabilities,
    dim=1
).item()

label_map = {
    "Non-toxic": 0,
    "Insults and Flaming": 1,
    "Other Offensive Texts": 2,
    "Hate and Harassment": 3,
    "Threats": 4,
    "Extremism": 5
}

print(probabilities)
print([key for key, val in label_map.items() if val == pred])

tensor([[0.1642, 0.7350, 0.0912, 0.0070, 0.0018, 0.0009]], device='cuda:0')
['Insults and Flaming']
